In [1]:
import pandas as pd
from pathlib import Path

# Definir os caminhos como objetos Path
arquivo_entrada = Path("06_PENSE2024_SEM_RESPOSTA_SEM_ABANDONO.parquet")
arquivo_saida = Path("07_PENSE2024_IDADE_13_A_17_ANOS.parquet")

# Carregar a base da etapa anterior
df = pd.read_parquet(arquivo_entrada)

# Definir a coluna de idade
coluna_idade = "IDADE DO ESTUDANTE, CATEGORIAS"

# Identificar registros com idade menor que 13 ou maior que 18
mascara_excluir = df[coluna_idade].astype("string").str.strip().isin(["1", "4"])

# Contabilizar registros
total_original = len(df)
total_excluir = mascara_excluir.sum()

# Criar nova base mantendo somente as categorias 2 e 3
df_idade = df.loc[~mascara_excluir].copy()

# Salvar a nova base
df_idade.to_parquet(
    arquivo_saida,
    index=False
)

# Exibir resumo do processamento
print("========== FILTRO DE IDADE CONCLUÍDO ==========")

print(f"Linhas originais: {total_original:,}")
print(f"Linhas removidas: {total_excluir:,}")
print(f"Linhas restantes: {len(df_idade):,}")

print(
    f"Percentual removido: "
    f"{total_excluir / total_original * 100:.2f}%"
)

mapa_idades = {
    "1": "Menos de 13 anos",
    "2": "13 a 14 anos",
    "3": "15 a 17 anos",
    "4": "18 anos ou mais",
    1: "13 a 14 anos",  # Tratamento caso os valores estejam como números inteiros
    2: "13 a 14 anos",
    3: "15 a 17 anos",
    4: "18 anos ou mais",
}

print("\nDistribuição das categorias de idade após o filtro:")

# Mapeia os valores mantendo a contagem e a ordenação
distribuicao = df_idade[coluna_idade].map(mapa_idades)
print(distribuicao.value_counts(dropna=False).sort_index())

print("\nArquivo salvo em:")
print(arquivo_saida)

print("\nArquivo existe:", arquivo_saida.exists())

# Definir o caminho completo do novo arquivo
arquivo_saida = Path("07_PENSE2024_IDADE_13_A_17_ANOS.parquet")

# Salvar a nova base em formato Parquet
df_idade.to_parquet(
    arquivo_saida,
    index=False
)

# Confirmar a gravação
print("Arquivo gravado com sucesso!")
print(f"Localização: {arquivo_saida}")
print(f"Total de registros: {len(df_idade):,}")
print(f"Arquivo existe: {arquivo_saida.exists()}")

========== FILTRO DE IDADE CONCLUÍDO ==========
Linhas originais: 142,929
Linhas removidas: 29,319
Linhas restantes: 113,610
Percentual removido: 20.51%

Distribuição das categorias de idade após o filtro:
IDADE DO ESTUDANTE, CATEGORIAS
13 a 14 anos    73765
15 a 17 anos    39845
Name: count, dtype: int64

Arquivo salvo em:
07_PENSE2024_IDADE_13_A_17_ANOS.parquet

Arquivo existe: True
Arquivo gravado com sucesso!
Localização: 07_PENSE2024_IDADE_13_A_17_ANOS.parquet
Total de registros: 113,610
Arquivo existe: True


In [2]:
import pandas as pd
from IPython.display import HTML, display

# 1. Carregar o dicionário de dados
dicionario = pd.read_excel("00_Dicionário_Microdados_PENSE2024.xlsx")

dico_map = {}
for _, row in dicionario.iterrows():
    desc = str(row["Descrição da Variável"]).strip()
    cat = str(row["Categoria"]) if pd.notna(row["Categoria"]) else ""
    cat_dict = {}
    if cat:
        for linha in cat.split("\n"):
            if " - " in linha:
                partes = linha.split(" - ", 1)
                cat_dict[partes[0].strip()] = partes[1].strip()
    dico_map[desc] = cat_dict

# 2. Gerar a tabela resumo para df_idade
tabela_dados = []

for col in df_idade.columns:
    counts = df_idade[col].astype(str).str.strip().value_counts(dropna=False)
    total = len(df_idade)
    cat_dict = dico_map.get(col, {})
    
    respostas_formatadas = []
    for cod, qtd in counts.items():
        rotulo = cat_dict.get(cod, "")
        pct = (qtd / total) * 100
        if rotulo:
            respostas_formatadas.append(f"[{cod}] {rotulo}: {pct:.2f}%")
        else:
            respostas_formatadas.append(f"[{cod}]: {pct:.2f}%")
            
    tabela_dados.append({
        "Pergunta / Variável": col,
        "Distribuição das Respostas": "\n".join(respostas_formatadas)
    })

df_tabela = pd.DataFrame(tabela_dados)

# 3. Estilizar e exibir
styler = df_tabela.style.set_properties(**{
    'white-space': 'pre-wrap',
    'text-align': 'left'
})

print("========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========\n")
display(HTML(styler.to_html()))

========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========



,Pergunta / Variável,Distribuição das Respostas
0,"NOS ÚLTIMOS 12 MESES, alguma vez você sentiu vontade de se machucar de propósito?",[2] Não: 67.16% [1] Sim: 32.84%
1,"NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que a vida não vale a pena ser vivida?",[1] Nunca: 48.42% [2] Raramente: 17.94% [3] Às vezes: 15.89% [4] Na maioria das vezes: 8.95% [5] Sempre: 8.79%
2,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu triste?",[3] Às vezes: 36.88% [2] Raramente: 23.71% [4] Na maioria das vezes: 19.89% [1] Nunca: 10.23% [5] Sempre: 9.29%
3,"NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que ninguém se preocupa com você?",[1] Nunca: 26.77% [3] Às vezes: 24.95% [2] Raramente: 22.83% [4] Na maioria das vezes: 14.55% [5] Sempre: 10.90%
4,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu muito preocupado com as coisas comuns do seu dia a dia como atividades da escola, competições esportivas, tarefas de casa, etc.?",[4] Na maioria das vezes: 28.95% [5] Sempre: 26.71% [3] Às vezes: 24.78% [2] Raramente: 11.04% [1] Nunca: 8.51%
5,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu irritado(a), nervoso(a) ou mal-humorado(a) por qualquer coisa?",[3] Às vezes: 32.26% [4] Na maioria das vezes: 25.96% [5] Sempre: 18.21% [2] Raramente: 15.28% [1] Nunca: 8.30%
6,Quantos(as) amigos(as) próximos você tem?,[4] 3 ou mais amigos: 75.47% [3] 2 amigos: 14.27% [2] 1 amigo: 6.50% [1] Nenhum amigo: 3.77%
7,"NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável entendeu suas preocupações e problemas?",[4] Na maioria das vezes: 22.26% [5] Sempre: 21.75% [3] Às vezes: 20.62% [2] Raramente: 17.95% [1] Nunca: 17.42%
8,"NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável sabia realmente o que você estava fazendo em seu tempo livre?",[4] Na maioria das vezes: 34.72% [5] Sempre: 33.69% [3] Às vezes: 15.07% [2] Raramente: 8.95% [1] Nunca: 7.57%
9,"NOS ÚLTIMOS 30 DIAS, quantas vezes algum dos seus colegas de escola o esculachou, zoou, mangou, intimidou ou caçoou tanto que você ficou magoado, incomodado, aborrecido, ofendido ou humilhado?",[1] Nenhuma vez nos últimos 30 dias: 59.61% [4] 3 ou mais vezes: 18.83% [2] 1 vez: 12.77% [3] 2 vezes: 8.78%
